# **Suite2p ↔ pClamp/wavesurfer (.abf) alignment**
### *Neural data ↔ ball locomotion + stimulus trials (entry / presentation / exit)*

- **Section 0** — dataset + channel map + stimulus log
- **Section 1** — load suite2p, compute **dff / z-dff / deconv**
- **Section 2** — align: **peak-detected galvo frames**, **parse stimulus trials**
  (entry→presentation→exit; double-click + estimated/aborted handled), attach the
  **stimulus log**, build the **trial dict + tensor**, export **trial-info CSV**,
  and map **neurons → cortical region**

> suite2p registration + cell curation (iscell) must be done first.
> The consolidated code lives in `abf_imaging_alignment.py` in this folder.

## **Section 0 : Config**

In [ ]:
import os, glob, pickle
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from abf_imaging_alignment import AlignmentSession   # same folder as this notebook

In [ ]:
# ---- WHICH SESSION (the only block you normally edit) -----------------------
experimenter = "Naomi"
mouse        = "GRINPL1"
date         = "2026-09-03"
root         = "/Volumes/Akhil Data/Naomi data"

# all paths follow from root / mouse / date (layout: see README_for_Akhil.txt)
abf_folder   = os.path.join(root, "Aligned", date)                # .abf files + logstimuli.csv
suite2p_path = os.path.join(root, "Suite2p", date, "plane0")
stim_log_csv = os.path.join(abf_folder, "logstimuli.csv")        # stimulus log
#roi_pickle   = os.path.join(os.path.dirname(suite2p_path), "..", "roi_indices_by_region.pkl")  # from Segment-ROIs notebook (optional)
save_path    = os.path.join(root, "ProcessedData", mouse, date)
for pth in (abf_folder, suite2p_path, stim_log_csv):
    assert os.path.exists(pth), f"missing: {pth}"

channels = {"galvo":6, "stim":4, "roll":0, "pitch":1, "yaw":2}   # analog-input index

# ---- parameters -------------------------------------------------------------
baseline_size      = 225      # dff baseline half-window (frames)
frame_method       = "peak"   # detect frames at the sawtooth PEAK
frames_per_tseries = 10000    # frames the scope saved per t-series (trims galvo warm-up)
frame_trim_side    = "start"  # drop leading warm-up cycles (keep last N)
long_thresh_s      = 2.0      # IN4 pulses longer than this = a PRESENTATION
n_frames_forward   = 375      # frames FORWARD from onset: 25 s (09-03 holds 15-25 s, onsets >= 28 s apart; 08-17 used 180 = 12 s)
pre_frames         = 60       # frames before onset (4 s at 15 Hz; entry TTL is ~1.7 s pre-onset)
light_correct      = True       # remove stray-light bleed (additive, same in every ROI)
dark_offset        = "auto"     # detector zero level subtracted from F/Fneu ("auto" = 1st pct of meanImg, or a number, or None)
dff_method         = "maximin"  # time-varying F0 (removes bleaching / drift / start-of-acquisition decay); "session" = old
f0_window_s        = 30.0       # maximin baseline window (s) — must be longer than responses (~8 s)
qc_min_f0          = 1.0        # drop cells whose baseline F0 ever falls to <= this (raw F units; dF/F undefined)
qc_max_noise       = None       # optional: drop cells with dF/F noise (robust SD per frame) above this
baseline_win_s     = (-2.0, 0.0)   # baseline window rel. onset (flat pre-onset stretch; entry TTL ~-1.8 s)

os.makedirs(save_path, exist_ok=True)
abf_paths = sorted(glob.glob(os.path.join(abf_folder, "*.abf")))
print(len(abf_paths), "abf files:", [os.path.basename(p) for p in abf_paths])
s = AlignmentSession(suite2p_path=suite2p_path, abf_paths=abf_paths, channels=channels)

## **Section 1 : DFF and Deconvolve**

In [ ]:
s.load_suite2p(use_iscell=True)
# dF/F pipeline: dark offset -> light-bleed removal -> neuropil subtraction (c=0.58)
# -> time-varying F0 (maximin, per acquisition) -> dF/F = (f - F0) / F0.
# Cache is parameter-aware (recomputes if any setting changes).
s.compute_activity(cache_path=os.path.join(save_path, "all_activity_py.pkl"),
                   method=dff_method, neuropil_coeff=0.58, deconvolve=True,
                   light_correct=light_correct, dark_offset=dark_offset, baseline_win_s=f0_window_s)
s.summary()

**1.1b – Cell QC.** Cells whose neuropil-corrected baseline F0 sits at the dark level have undefined
(exploding) dF/F. They are dropped here, before trials and tensors are built, and every decision is
saved to `cell_qc.csv` (`neuron` = index in Naomi's iscell list, `roi` = suite2p ROI).

In [ ]:
qc = s.cell_qc(min_f0=qc_min_f0, max_noise=qc_max_noise)
qc.to_csv(os.path.join(save_path, "cell_qc.csv"), index=False)
display(qc[~qc.keep]) if (~qc.keep).any() else print("all cells pass QC")
s.apply_cell_qc(qc.keep.to_numpy())

#### (optional) OASIS deconvolution — needs the `oasis` package

In [ ]:
# Deconvolution runs inside compute_activity (deconvolve=True) and is cached —
# nothing to do here. (Set deconvolve=False above if you do not need spikes.)

In [ ]:
# all_activity_py.pkl is written by compute_activity (the cache) — nothing to save here.

## **Section 2 : Align to behavior**

**2.1 – Frames + stimulus trials.** Frames are detected at the galvo sawtooth
**peaks** and reconciled to `frames_per_tseries`. `parse_stim` builds
entry/presentation/exit trials (double-click entries kept; a missing presentation
pulse gets an onset estimated from entry + median delay). `attach_stim_log`
assigns each trial its stimulus name.

In [ ]:
s.load_abf()
s.detect_frames(method=frame_method, frames_per_tseries=frames_per_tseries, trim_side=frame_trim_side)
s.detect_stim()
s.parse_stim(long_thresh_s=long_thresh_s)
s.attach_stim_log(stim_log_csv)
s.summary()
if s.stim_log_report: print("MISMATCHES:", s.stim_log_report)

**2.2 – QC:** frame peaks + trial structure (entry green, presentation blue,
estimated orange-dashed, exit red).

In [ ]:
#fig,ax=plt.subplots(figsize=(12,3)); s.plot_frame_qc(acq=0, window_s=(100,101.5), ax=ax); plt.show()
#fig,ax=plt.subplots(figsize=(14,3)); s.plot_trials(acq=0, ax=ax); plt.show()

**2.3 – Validate** detected frames against suite2p `frames_per_folder`.

In [ ]:
report = s.validate(cap=True, trim_side=frame_trim_side); report

**2.4 – Trial-by-trial dict** (one entry per PRESENTATION; neural window runs
FORWARD `n_frames_forward` from the presentation-onset frame).

In [ ]:
trials = s.build_trials(n_frames=n_frames_forward, pre_frames=pre_frames)
print(len(trials), "presentation trials")
pickle.dump(trials, open(os.path.join(save_path,"imaging_trials_py.pkl"),"wb"))

**2.5 – Trial-info CSV.** One row per trial (`tensor_index` links each row to
the tensor); aborted trials are logged with a blank `tensor_index`.

In [ ]:
trial_df = s.save_trials_csv(os.path.join(save_path,"trials_info.csv"))
trial_df.head()

In [ ]:
trial_df['stim']

**2.6 – Presentation-locked tensor** `(n_trials, n_neurons, n_frames)` with a
stimulus label per trial (`s.epoch_labels`).

In [ ]:
# Build tensors for all three activity signals
# epoch returns (n_trials, n_neurons, pre_frames + n_frames_forward)
X_zdff_raw   = s.epoch("z_dff",  n_frames=n_frames_forward, pre_frames=pre_frames)
X_dff_raw    = s.epoch("dff",    n_frames=n_frames_forward, pre_frames=pre_frames)
X_deconv_raw = s.epoch("deconv", n_frames=n_frames_forward, pre_frames=pre_frames)
labels       = np.array(s.epoch_labels)
assert all(l is not None for l in labels), f"unlabelled trials at {np.where(labels == None)[0]}"

# Per-trial baseline correction: subtract the mean of the baseline_win_s window
# Keep the full trace (pre + post) so baseline period is available for plotting.
fs = s.suite2p["fs"]
bl_a = pre_frames + int(round(baseline_win_s[0] * fs))
bl_b = pre_frames + int(round(baseline_win_s[1] * fs))
def baseline_correct(X, a, b):
    bl = np.nanmean(X[:, :, a:b], axis=2, keepdims=True)  # (trials, neurons, 1)
    return X - bl

X_zdff   = baseline_correct(X_zdff_raw,   bl_a, bl_b)
X_dff    = baseline_correct(X_dff_raw,    bl_a, bl_b)
X_deconv = baseline_correct(X_deconv_raw, bl_a, bl_b)

print("z_dff tensor:", X_zdff.shape, f"  (onset at frame {pre_frames}; baseline frames {bl_a}:{bl_b})")
print("dff tensor:  ", X_dff.shape)
print("deconv tensor:", X_deconv.shape)
print("trials with NaN (window past end of imaging):", int(np.isnan(X_zdff).any(axis=(1, 2)).sum()))
print("stimuli:", sorted(set(labels)))

np.save(os.path.join(save_path, "tensor_zdff.npy"),   X_zdff)
np.save(os.path.join(save_path, "tensor_dff.npy"),    X_dff)
np.save(os.path.join(save_path, "tensor_deconv.npy"), X_deconv)
np.save(os.path.join(save_path, "tensor_labels.npy"), labels)
# uncorrected tensors: analysis code can choose its own baseline (or none)
np.save(os.path.join(save_path, "tensor_zdff_raw.npy"),   X_zdff_raw)
np.save(os.path.join(save_path, "tensor_dff_raw.npy"),    X_dff_raw)
np.save(os.path.join(save_path, "tensor_deconv_raw.npy"), X_deconv_raw)
# window metadata so analysis notebooks can't get pre_frames / fs out of sync
np.save(os.path.join(save_path, "tensor_meta.npy"),
        {"pre_frames": pre_frames, "n_frames_forward": n_frames_forward, "fs": float(fs),
         "baseline_win_s": baseline_win_s, "mouse": mouse, "date": date,
         "suite2p_path": suite2p_path, "n_cells": int(X_dff.shape[1]),
         "cell_idx": [int(i) for i in s.suite2p["cell_idx"]]})   # suite2p ROI of each tensor neuron

**2.6b – Light-bleed QC.** Stray light adds the same absolute fluorescence to every ROI, so the
frame-wise median of all ROIs' neuropil traces tracks it. Top: the light trace over the session.
Bottom: light trace locked to presentation onset (mean ± SD across trials). After correction this
trace has been subtracted from every F and Fneu.

In [ ]:
if s.light is not None:
    np.save(os.path.join(save_path, "light_trace.npy"), s.light)
    fs = s.suite2p["fs"]
    fig, axs = plt.subplots(2, 1, figsize=(10, 4.5))
    axs[0].plot(np.arange(len(s.light)) / fs / 60, s.light, lw=0.4, color="k")
    for e in np.cumsum(s.suite2p["frames_per_folder"])[:-1]:
        axs[0].axvline(e / fs / 60, color="r", lw=0.6, ls="--")
    axs[0].set(xlabel="session time (min; red = new acquisition)", ylabel="light (raw F units)")
    W = np.arange(-16 * int(fs), 12 * int(fs))
    on = trial_df["onset_frame"].dropna().to_numpy(int)
    seg = np.stack([s.light[o + W] for o in on if o + W[0] >= 0 and o + W[-1] < len(s.light)])
    tt = W / fs
    axs[1].plot(tt, seg.mean(0), color="k"); axs[1].fill_between(tt, seg.mean(0) - seg.std(0), seg.mean(0) + seg.std(0), color="0.8")
    axs[1].axvline(0, color="k", ls="--", lw=0.8)
    axs[1].set(xlabel="time from presentation onset (s)", ylabel="light (raw F units)")
    fig.tight_layout(); plt.show()
else:
    print("light correction off (light_correct=False)")

In [ ]:
# Save session-length ball signals (aligned to imaging frames) for kinematics regression
ball = s.ball_session()
np.savez(os.path.join(save_path, "ball_session.npz"), **ball)
print("ball signals:", {k: v.shape for k, v in ball.items()})

# Save session-level continuous signals for kinematics regression
np.save(os.path.join(save_path, "session_zdff.npy"), s.z_dff)
np.save(os.path.join(save_path, "session_dff.npy"), s.dff)
print("session z_dff:", s.z_dff.shape, "| dff:", s.dff.shape)

**2.7 – Map neurons → cortical region** (from the *Segment ROIs by Anatomical
Region* notebook). `regions` is aligned to the tensor's neuron axis.

In [ ]:
try:
    roi_dict = pickle.load(open(os.path.abspath(roi_pickle), "rb"))
    regions = s.neuron_regions(roi_dict, s.suite2p["cell_idx"])
    np.save(os.path.join(save_path,"neuron_regions.npy"), regions)
    print({r:int((regions==r).sum()) for r in sorted(set(regions))})
    # e.g. slice the tensor by area:  X_v1 = X[:, regions=="V1", :]
except (FileNotFoundError, NameError):   # NameError: roi_pickle not set (small FOV)
    print("roi_indices_by_region.pkl not found — run the Segment-ROIs notebook first")

**2.8 – Quick sanity: mean response per stimulus**

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
fs = s.suite2p["fs"]
t = (np.arange(pre_frames + n_frames_forward) - pre_frames) / fs  # negative = baseline
for stim in sorted(set(labels)):
    m = np.nanmean(X_zdff[labels==stim], axis=(0,1))
    ax.plot(t, m, label=f"{stim} (n={np.sum(labels==stim)})")
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set(xlabel="time from presentation onset (s)", ylabel="mean z-dff", title="response by stimulus")
ax.legend(fontsize=8); plt.show()